# Формы импульсов: $^{60}$Co и $^{252}$Cf

Ноутбук случайно выбирает события одного канала и рисует формы импульсов для двух экспериментальных случаев: чистая гамма (`60Co`) и смесь нейтронов с гамма (`252Cf`). Все пути вычисляются относительно корня проекта.

In [ ]:
from pathlib import Path
import ROOT
import numpy as np
import matplotlib.pyplot as plt

ROOT.gROOT.SetBatch(True)
plt.style.use('seaborn-v0_8-whitegrid')

# Параметры просмотра
CHANNEL = 0                 # доступны 0, 2, 3, 4, 5
N_PULSES = 10              # число импульсов для каждого источника
ENERGY_RANGE = (0, 65535)  # отбор по ветви Energy; например (500, 3000)
SUBTRACT_BASELINE = True   # вычитать медиану первых BASELINE_SAMPLES отсчётов
BASELINE_SAMPLES = 30
SEED = 42                  # измените seed, чтобы увидеть другие события


In [ ]:
def find_project_root():
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        if (folder / 'gamma_n_data').is_dir():
            return folder
    raise FileNotFoundError('Не найдена директория gamma_n_data')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'gamma_n_data'

files = {
    '60Co — гамма': DATA_DIR / 'call_all_60Co' / 'UNFILTERED' / f'Data_CH{CHANNEL}@DT5730SB_27616_call_all_60Co.root',
    '252Cf — нейтроны + гамма': DATA_DIR / 'call_all_252Cf' / 'UNFILTERED' / f'Data_CH{CHANNEL}@DT5730SB_27616_call_all_252Cf.root',
}

for label, path in files.items():
    if not path.is_file():
        raise FileNotFoundError(f'{label}: {path}')
    print(f'{label}: {path.relative_to(PROJECT_ROOT)}')

In [ ]:
def load_random_pulses(path, count=10, energy_range=(0, 65535), seed=42):
    root_file = ROOT.TFile.Open(str(path), 'READ')
    if not root_file or root_file.IsZombie():
        raise OSError(f'Не удалось открыть {path}')
    tree = root_file.Get('Data')
    if not tree:
        root_file.Close()
        raise KeyError(f'В {path.name} нет дерева Data')

    # Читаем только нужные ветви: это заметно быстрее для больших файлов.
    tree.SetBranchStatus('*', False)
    tree.SetBranchStatus('Energy', True)
    tree.SetBranchStatus('Samples', True)
    n_entries = int(tree.GetEntries())
    rng = np.random.default_rng(seed)
    max_candidates = min(n_entries, max(10_000, count * 100))
    candidates = rng.choice(n_entries, size=max_candidates, replace=False)

    pulses, energies, entries = [], [], []
    e_min, e_max = energy_range
    for entry in candidates:
        tree.GetEntry(int(entry))
        energy = int(tree.Energy)
        if e_min <= energy <= e_max:
            pulses.append(np.array(tree.Samples, dtype=float))
            energies.append(energy)
            entries.append(int(entry))
            if len(pulses) == count:
                break
    root_file.Close()
    if len(pulses) < count:
        raise ValueError(
            f'Найдено только {len(pulses)} событий в диапазоне {energy_range}. '
            'Расширьте ENERGY_RANGE.'
        )
    return pulses, np.asarray(energies), np.asarray(entries)


def remove_baseline(pulse, n_samples=30):
    n = min(n_samples, len(pulse))
    return pulse - np.median(pulse[:n])

In [ ]:
datasets = {}
for offset, (label, path) in enumerate(files.items()):
    datasets[label] = load_random_pulses(
        path, count=N_PULSES, energy_range=ENERGY_RANGE, seed=SEED + offset
    )

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True, constrained_layout=True)
colors = ['tab:blue', 'tab:orange']

for ax_index, ((label, (pulses, energies, entries)), color) in enumerate(zip(datasets.items(), colors)):
    for pulse, energy, entry in zip(pulses, energies, entries):
        y = remove_baseline(pulse, BASELINE_SAMPLES) if SUBTRACT_BASELINE else pulse
        axes[ax_index].plot(y, alpha=0.75, lw=1.1, label=f'entry {entry}, E={energy}')
    axes[ax_index].set_title(f'{label} — {len(pulses)} импульсов')
    axes[ax_index].set_xlabel('Номер отсчёта')
    axes[ax_index].set_ylabel('ADC − baseline' if SUBTRACT_BASELINE else 'ADC')
    axes[ax_index].legend(fontsize=7, ncol=2)

fig.suptitle(f'Канал {CHANNEL}; Energy {ENERGY_RANGE[0]}–{ENERGY_RANGE[1]}', fontsize=15)
plt.show()

Чтобы посмотреть другие импульсы, измените `SEED`. Для сравнения в одном энергетическом диапазоне задайте, например, `ENERGY_RANGE = (500, 3000)`. После изменения параметров выполните все ячейки заново (`Run → Run All Cells`).